# Homework 13 — Sampling and Variational Approximation

**Coverage:** Lectures 27–28  
**Due:** Sunday, December 6, 2026, 11:59 p.m. ET  
**Total:** 100 points

## Instructions

- Complete this notebook in Google Colab.
- Problem 1 is a manual mathematics problem. Show every important
  step in Markdown/LaTeX, or insert one clearly legible image of
  your handwritten derivation. Code may check arithmetic only
  after the derivation is complete.
- Problem 2 is a scaffolded scientific-computing study. Use the
  supplied random seeds and do not delete setup, helper, or check
  cells.
- Your submitted notebook must run from beginning to end in a
  fresh Colab runtime without Google Drive, absolute paths, or
  additional package installation.
- Label plots and include documented units. If a legacy dataset
  has no documented units, label the quantity as normalized or
  unit-unspecified rather than inventing units. Unless stated
  otherwise, report numerical answers to at least four
  significant digits.

## Student details

- **First name:**
- **Last name:**
- **Purdue email:**


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from numpy.polynomial.hermite import hermgauss
from scipy.optimize import minimize
from scipy.special import expit, logsumexp

SEED = 53913
rng = np.random.default_rng(SEED)
sns.set_theme(style="ticks", context="notebook")
plt.rcParams["figure.dpi"] = 120
np.set_printoptions(precision=6, suppress=True)

# During drafting this points to master. Before release, the instructor
# will replace DATA_REVISION with the immutable course release tag.
DATA_REVISION = "master"
DATA_BASE = (
    "https://raw.githubusercontent.com/PredictiveScienceLab/"
    f"data-analytics-se/{DATA_REVISION}/lecturebook/data/homework"
)

def course_data(name):
    local_candidates = [
        Path("../data/homework") / name,
        Path("lecturebook/data/homework") / name,
    ]
    for local in local_candidates:
        if local.exists():
            return local
    return f"{DATA_BASE}/{name}"


## Problem 1 — Two approximations to a correlated Gaussian (25 points)

Let \(p(\theta)=N(0,\Sigma)\),
\(\Sigma=\begin{bmatrix}1&.8\\.8&1\end{bmatrix}\). A Metropolis
proposal is \(N(\theta,s^2I)\), and
\(q=N(m,D)\) with diagonal \(D=\operatorname{diag}(d_1,d_2)\).

1. Derive the Metropolis acceptance probability and identify the
   cancellations. **(5)**
2. Evaluate it from \((0,0)^T\) to \((1,0)^T\). **(4)**
3. Write \(D_{KL}(q\|p)\) in closed form. **(6)**
4. Minimize over \(m,d_1,d_2\), deriving
   \(m=0,d_i=1/(\Sigma^{-1})_{ii}\), and evaluate. **(7)**
5. Explain the lost dependence and marginal underdispersion. **(3)**


> **Response:** Replace this text with your work.


## Problem 2 — Challenger logistic-posterior inference (75 points)

The frozen teaching extract has 23 pre-accident shuttle launches
with nonmissing damage indicators. Model
\[
x_i=(T_i-70)/10,\qquad
y_i\sim\operatorname{Bernoulli}[\operatorname{logit}^{-1}
(\alpha+\beta x_i)],
\]
with independent \(\alpha,\beta\sim N(0,2.5^2)\).


In [ ]:
challenger = pd.read_csv(course_data("hw13_challenger_prelaunch.csv"))
temperature = challenger["temperature_f"].to_numpy(float)
y = challenger["damage_incident"].to_numpy(int)
x = (temperature - 70.0) / 10.0
PRIOR_SD = 2.5
assert len(challenger) == 23 and set(np.unique(y)) <= {0, 1}
challenger.head()


In [ ]:
def log_posterior(theta):
    '''Stable, vectorized log posterior up to a constant.'''
    theta = np.asarray(theta, float)
    eta = theta[..., 0, None] + theta[..., 1, None] * x
    log_likelihood = np.sum(
        y * eta - np.logaddexp(0.0, eta), axis=-1
    )
    log_prior = -0.5 * np.sum((theta / PRIOR_SD)**2, axis=-1)
    return log_likelihood + log_prior

assert np.isclose(log_posterior([0.0, 0.0]), -15.9423851529)
assert np.isclose(log_posterior([-1.0, -2.0]), -10.6419612123)

def numerical_hessian(function, point, step=1e-4):
    '''Central-difference Hessian for this two-parameter problem.'''
    point = np.asarray(point, float)
    dimension = len(point)
    H = np.empty((dimension, dimension))
    for i in range(dimension):
        ei = np.zeros(dimension); ei[i] = step
        for j in range(dimension):
            ej = np.zeros(dimension); ej[j] = step
            H[i, j] = (
                function(point + ei + ej)
                - function(point + ei - ej)
                - function(point - ei + ej)
                + function(point - ei - ej)
            ) / (4 * step**2)
    return 0.5 * (H + H.T)

GRID_SIZE = 301

def normalized_reference_grid(logp, center, covariance,
                              grid_size=GRID_SIZE):
    '''Normalize a MAP ± 6 marginal-SD rectangular grid.

    Boundary mass is the sum of normalized discrete weights on
    the outermost rows and columns, counting corners once.
    '''
    sd = np.sqrt(np.diag(covariance))
    axes = [
        np.linspace(center[j] - 6*sd[j], center[j] + 6*sd[j], grid_size)
        for j in range(2)
    ]
    A, B = np.meshgrid(*axes, indexing="ij")
    points = np.column_stack([A.ravel(), B.ravel()])
    log_weights = logp(points).reshape(A.shape)
    weights = np.exp(log_weights - logsumexp(log_weights))
    edge = np.zeros_like(weights, dtype=bool)
    edge[[0, -1], :] = True; edge[:, [0, -1]] = True
    return axes, weights, float(weights[edge].sum())

def component_ess(samples):
    '''Initial-positive-sequence ESS for every sample column.'''
    samples = np.asarray(samples, float)
    n = len(samples)
    result = []
    for values in samples.T:
        centered = values - values.mean()
        size = 1 << (2*n - 1).bit_length()
        spectrum = np.fft.rfft(centered, n=size)
        acov = np.fft.irfft(spectrum * spectrum.conjugate(), n=size)[:n]
        if acov[0] <= 0:
            result.append(float(n))
            continue
        acf = acov / acov[0]
        pair_count = len(acf) // 2
        paired = acf[:2*pair_count].reshape(-1, 2).sum(axis=1)
        first_nonpositive = np.flatnonzero(paired <= 0)
        keep = (
            first_nonpositive[0]
            if len(first_nonpositive)
            else len(paired)
        )
        tau = max(1.0, -1.0 + 2.0 * paired[:keep].sum())
        result.append(n / tau)
    return np.asarray(result)

def random_walk_metropolis(logp, initial, proposal_chol,
                           n_steps, local_rng):
    '''Complete only the marked log-scale acceptance decision.'''
    theta = np.asarray(initial, float).copy()
    current_lp = float(logp(theta))
    samples = np.empty((n_steps, len(theta)))
    accepted = 0
    for i in range(n_steps):
        proposed = theta + proposal_chol @ local_rng.normal(size=len(theta))
        proposed_lp = float(logp(proposed))
        accept = False  # TODO: replace with the log-scale MH test
        if accept:
            theta, current_lp = proposed, proposed_lp
            accepted += 1
        samples[i] = theta
    return samples, accepted / n_steps

gh_nodes, gh_weights = hermgauss(20)
GH_Z = np.stack(
    np.meshgrid(gh_nodes, gh_nodes, indexing="ij"), axis=-1
).reshape(-1, 2)
GH_W = np.outer(gh_weights, gh_weights).ravel() / np.pi

def vi_objective(parameters):
    mean = parameters[:2]
    log_sd = parameters[2:]
    sd = np.exp(log_sd)
    theta = mean + np.sqrt(2.0) * sd * GH_Z
    expected_logp = GH_W @ log_posterior(theta)
    expected_logq = -1.0 * (1.0 + np.log(2*np.pi)) - log_sd.sum()
    return expected_logq - expected_logp


### 2.1 Posterior and deterministic reference (10 points)

Verify the supplied stable log posterior and explain the temperature
rescaling and prior. Find the MAP with BFGS and use the supplied
Hessian helper for the Laplace covariance. Normalize the supplied
301-by-301 grid spanning MAP plus/minus six Laplace marginal SDs.
Report the helper's explicitly defined outer-edge mass and require
it to be below \(10^{-4}\).


In [ ]:
# YOUR CODE HERE


### 2.2 Random-walk Metropolis (20 points)

Implement the acceptance decision on the log scale. Use the
Laplace Cholesky as proposal preconditioner; start every chain at
the MAP and compare scalar scales 0.8, 1.2, and 1.5 in 3,000-step
pilots. Create four independent generators with
`np.random.SeedSequence(53913).spawn(4)`, using the first three in
scale order and the fourth for the final run. Among pilots with
acceptance in `[0.20, 0.60]`, select the largest minimum-component
ESS from the supplied helper. Run 25,000 iterations, discard 5,000,
and report traces, acceptance, and component ESS.


In [ ]:
# YOUR CODE HERE


### 2.3 Diagonal Gaussian variational inference (20 points)

Parameterize \(q=N(m,\operatorname{diag}(e^{2\ell_1},e^{2\ell_2}))\).
Use the supplied 20-by-20 Gauss–Hermite objective and minimize
\(E_q[\log q-\log p]\) using L-BFGS-B, bounds
\(-5\le\ell_i\le2\), initialized at the MAP and Laplace marginal
SDs. Verify optimizer convergence and numerical stability.


In [ ]:
# YOUR CODE HERE


### 2.4 Compare the approximations (15 points)

Compare grid, MCMC, and VI means, marginal 95% intervals,
covariance matrices, and two-dimensional contours. Explicitly
identify which dependence the diagonal approximation cannot
represent and where underdispersion appears.


In [ ]:
# YOUR CODE HERE


### 2.5 Predictive risk and extrapolation (10 points)

Using retained MCMC draws as the primary approximation, plot the
pointwise median and 95% credible band for damage probability over
the observed temperature range. At \(31^\circ\mathrm F\), report
the median and central 95% interval under the grid reference,
MCMC, and VI. Explain why this is hazardous
extrapolation, not an established causal fact, and why discarding
covariance can affect this derived probability differently from
parameter marginal variances.


In [ ]:
# YOUR CODE HERE


> **Response:** Replace this text with your work.
